# Laser Execute

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


In [ ]:
from femtum_sdk.api import InstrumentApi
from femtum_sdk.api.contracts.v1.instruments.health_pb2 import HealthRequest
from femtum_sdk.api.trimming import TrimmingApi

api_host = "localhost:5210"

# The trimming services are served by the instrument daemon, which owns the health endpoint.
with InstrumentApi(host=api_host) as api:
    health_response = api.health.GetHealth(HealthRequest())
    print(health_response)


## Execution Simulation

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure, DistanceMeasureUnit, EnergyMeasure, EnergyMeasureUnit
from femtum_sdk.api.contracts.v1.trimming.laser_execution_pb2 import LaserExecutionDistance, SimulateDatabaseRecipeRequest, SimulateDatabaseRecipeResponse, LaserExecutionOptions


with TrimmingApi(host=api_host) as api:
    simulation_iteration_1_response: SimulateDatabaseRecipeResponse = api.laser_execution.SimulateDatabaseRecipe(SimulateDatabaseRecipeRequest(
        recipe_id="4",
        options=LaserExecutionOptions(
            laser_energy_calibration_id="3",
            distance=LaserExecutionDistance(
                relative_from_previous_execution=DistanceMeasure(
                    value=0.0,
                    unit=DistanceMeasureUnit.Microns,
                ),
            ),
            energy_per_pulse=EnergyMeasure(
                value=0.2,
                unit=EnergyMeasureUnit.Nanojoules,
            ),
        ),
        previous_executions=[],
    ))

    simulation_iteration_2_response: SimulateDatabaseRecipeResponse = api.laser_execution.SimulateDatabaseRecipe(SimulateDatabaseRecipeRequest(
        recipe_id="4",
        options=LaserExecutionOptions(
            laser_energy_calibration_id="3",
            distance=LaserExecutionDistance(
                relative_from_previous_execution=DistanceMeasure(
                    value=10.0,
                    unit=DistanceMeasureUnit.Microns,
                ),
            ),
            energy_per_pulse=EnergyMeasure(
                value=0.2,
                unit=EnergyMeasureUnit.Nanojoules,
            ),
        ),
        previous_executions=simulation_iteration_1_response.executions,
    ))

    print(simulation_iteration_2_response.executions)



## Execute With Laser

In [ ]:
from femtum_sdk.api.contracts.v1.trimming.laser_execution_pb2 import ExecuteDatabaseRecipeRequest, ExecuteDatabaseRecipeResponse


with TrimmingApi(host=api_host) as api:
    response: ExecuteDatabaseRecipeResponse = api.laser_execution.ExecuteDatabaseRecipe(ExecuteDatabaseRecipeRequest(
        recipe_id="4",
        component_path="MyComponent",
        options=LaserExecutionOptions(
            laser_energy_calibration_id="3",
            distance=LaserExecutionDistance(
                relative_from_previous_execution=DistanceMeasure(
                    value=0.0,
                    unit=DistanceMeasureUnit.Microns,
                ),
            ),
            energy_per_pulse=EnergyMeasure(
                value=0.2,
                unit=EnergyMeasureUnit.Nanojoules,
            ),
        ),
    ))
    execution = response.execution

## Fetch executions

In [ ]:
from femtum_sdk.api import DatastoreApi
from femtum_sdk.api.contracts.v1.storage.laser_execution_pb2 import LaserExecutionPage, LaserExecutionsFilterRequest, ListByPageLaserExecutionsRequest
from femtum_sdk.api.contracts.v1.storage.query_pb2 import QueryField, QueryOperator


datastore_host = "localhost:5208"

with DatastoreApi(host=datastore_host) as datastore_api:
    is_up = datastore_api.health.is_up()
    print(f"Is Datastore API Up: {is_up}")
    assert is_up

    page: LaserExecutionPage = datastore_api.laser_execution.ListByPage(ListByPageLaserExecutionsRequest(filters=LaserExecutionsFilterRequest(component_path=QueryField(value="MyComponent", operator=QueryOperator.EQUALS))))
    assert any(item.id == execution.id for item in page.items)